# 07 — Paired paraphrase design

Builds the paired (source, paraphrase) subset the corpus does not contain, and scores each
document before and after paraphrasing with the source held fixed.

Section 3 first confirms that no pairing can be recovered from the corpus: the maximum exact
Jaccard similarity from any AI-obfuscated document to any human or AI-generated document is
0.0. Route A therefore finds nothing and Route B generates the pairs.

- Input: `data/cleaned/*.json`, weights from notebook 09
- Output: `results_r2_paired/table_paired_paraphrase.csv`, `transition_*.csv`,
  `paired_generations.jsonl`, `fig_paired_paraphrase.png`
- Paraphraser: `Qwen/Qwen2.5-7B-Instruct` (same family as the corpus generator — this
  isolates paraphrasing with generator family held constant, and is not an unseen-paraphraser test)
- Runtime: ~1 h on an A100, resumable
- Reported in: paper Section VI-H

Run notebook 09 first: this notebook loads the weights it saves.

## 1. Config

In [ ]:
# CONFIG — the only cell you should need to edit
REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2_paired'

# ---- Route A: recover native (source, paraphrase) pairs ---------------
RUN_ROUTE_A        = True
TQ_PARAPHRASE_FILE = ''                 # <-- EDIT: path to TQ_SYNT_Identification_of_paraphrasing.*
SRC_FIELD          = 'Paraphrase_text1' # <-- VERIFY against the printed column list
TGT_FIELD          = 'Paraphrase_text2'

# ---- Route B: generate the paired triples -----------------------------
RUN_ROUTE_B    = True
N_PER_ARM      = 300          # human docs and AI-generated docs to paraphrase
SAMPLE_FROM    = 'test'       # keep to the held-out partition
PARAPHRASER_ID = 'Qwen/Qwen2.5-7B-Instruct'   # <-- EDIT: an HF instruct model you can access
LOAD_IN_4BIT   = True         # required to fit an 8-9B model on a Colab T4
GEN_BATCH      = 8
TEMPERATURE    = 0.8
TOP_P          = 0.95
SEED           = 42

# Token budget. A FIXED new-token cap truncates long paraphrases and biases the
# Δ-words statistic downwards — which is the very quantity this notebook measures.
NEW_TOKEN_MULTIPLIER = 1.8    # new-token budget = multiplier * longest source in batch + slack
NEW_TOKEN_SLACK      = 64
MAX_NEW_TOKENS_CAP   = 768    # hard ceiling, for memory
MAX_SRC_TOKENS       = 896    # sources longer than this are EXCLUDED, never truncated

# ---- Classifier used to score the pairs -------------------------------
# Reuse the weights saved by notebook 09 if available; otherwise train here AND SAVE.
CLF_MODEL        = 'mDeBERTa-v3 (base)'
CLF_FAMILY       = 'deberta_v3'
CLF_PRESET       = 'deberta_v3_base_multi'
CLF_SEEDS        = [42, 1, 2]
WEIGHTS_DIR      = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2_seeds/weights'
TRAIN_IF_MISSING = False      # a model trained here is not the model the paper reports
SEQ_LEN = 256; EPOCHS = 5; BATCH_SIZE = 16; LEARNING_RATE = 2e-5; WEIGHT_DECAY = 0.01
PREDICT_BATCH = 64

# ---- Validation of generations ---------------------------------------
MIN_PARAPHRASE_WORDS = 5
MIN_CYRILLIC_RATIO   = 0.50   # over alphabetic characters only
DROP_LOW_CYRILLIC    = True
VERBATIM_JACCARD     = 0.95   # 5-gram overlap at or above this = not a paraphrase
BOOTSTRAP_N          = 5000

# ---- Environment ------------------------------------------------------
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers', 'accelerate', 'bitsandbytes', 'sentencepiece',
                'keras>=3.3', 'keras-hub', 'scikit-learn', 'scipy', 'statsmodels',
                'pandas', 'matplotlib'], check=False)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as _e:
    print('Not on Colab or Drive already mounted (%s)' % type(_e).__name__)

# Auto-load the shared paths written by notebook 00_setup_and_check.
# If r2_config.json is absent, the values set above are used unchanged.
try:
    import json as _json
    from pathlib import Path as _Path
    _cfg_path = _Path('/content/drive/MyDrive/r2_config.json')
    if _cfg_path.exists():
        _cfg = _json.load(open(_cfg_path))
        REPO_DIR    = _cfg.get('REPO_DIR', REPO_DIR)
        RESULTS_DIR = _cfg.get('RESULTS_PAIRED_DIR', RESULTS_DIR)
        WEIGHTS_DIR = _cfg.get('WEIGHTS_DIR', WEIGHTS_DIR)
        print('Paths loaded from r2_config.json')
        for _k, _v in [('REPO_DIR', REPO_DIR), ('RESULTS_DIR', RESULTS_DIR),
                       ('WEIGHTS_DIR', WEIGHTS_DIR)]:
            print(f'  {_k:12s}= {_v}')
    else:
        print('r2_config.json not found -- using the paths set above. '
              'Run 00_setup_and_check.ipynb to generate it.')
except Exception as _e:
    print('Could not load r2_config.json (%s: %s) -- using the paths set above.'
          % (type(_e).__name__, _e))


## 2. Imports and data

In [ ]:
import os, json, re, gc, time, platform, warnings
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

T_START = time.time()
REPO = Path(REPO_DIR)
OUT  = Path(RESULTS_DIR); OUT.mkdir(parents=True, exist_ok=True)
Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}
LABELS   = [0, 1, 2]

def load_split(name):
    """Load one cleaned split and fail loudly if the contract is broken."""
    p = REPO / f'data/cleaned/{name}_cleaned.json'
    if not p.exists():
        raise FileNotFoundError(
            f'{p} not found. Check REPO_DIR, and that Drive is mounted.')
    d = pd.DataFrame(json.load(open(p, encoding='utf-8')))
    missing = {'text', 'label'} - set(d.columns)
    if missing:
        raise KeyError(f'{p.name} is missing column(s) {sorted(missing)}; '
                       f'columns present: {list(d.columns)}')
    d['text']   = d['text'].astype(str).str.strip()
    d['label']  = d['label'].astype(int)
    d['split']  = name
    d['n_words'] = d['text'].str.split().str.len()
    d = d.reset_index(drop=True)
    d['doc_id'] = [f'{name}_{i}' for i in range(len(d))]
    bad = sorted(set(d['label']) - set(LABELS))
    if bad:
        raise ValueError(f'{p.name} contains unexpected label(s) {bad}; expected {LABELS}')
    return d

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
print(f'loaded  train={len(train)}  dev={len(dev)}  test={len(test)}')

balance = pd.DataFrame({s: d['label'].map(ID2LABEL).value_counts()
                        for s, d in [('train', train), ('dev', dev), ('test', test)]}).fillna(0).astype(int)
print('\nclass balance')
print(balance.to_string())
print('\nmedian length (words) by class, test split')
print(test.groupby(test.label.map(ID2LABEL))['n_words'].median().to_string())
print(f'\npython {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')


## 3. Confirm the corpus is unpaired (exact Jaccard, with positive control)

In [ ]:
K_SHINGLE = 5

def shingle_set(text, k=K_SHINGLE):
    """Word k-gram shingles. Documents shorter than k contribute one whole-text shingle."""
    toks = re.findall(r'\w+', str(text).lower(), flags=re.UNICODE)
    if not toks:
        return frozenset()
    if len(toks) < k:
        return frozenset([' '.join(toks)])
    return frozenset(' '.join(toks[i:i + k]) for i in range(len(toks) - k + 1))

def jaccard(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)

allsp = pd.concat([train, dev, test], ignore_index=True)
shs   = [shingle_set(t) for t in allsp['text']]
lab   = allsp['label'].to_numpy()

obf_idx = np.flatnonzero(lab == 2)
ref_idx = np.flatnonzero(lab != 2)
print(f'{len(obf_idx)} AI-obfuscated documents scanned against '
      f'{len(ref_idx)} human / AI-generated documents')

# inverted index: shingle -> reference documents containing it
inv = defaultdict(list)
for j in ref_idx:
    for s in shs[j]:
        inv[s].append(int(j))
inv = dict(inv)
print(f'inverted index: {len(inv):,} distinct 5-grams')

best_jac  = np.zeros(len(obf_idx))
best_con  = np.zeros(len(obf_idx))
best_mate = np.full(len(obf_idx), -1)
n_cand    = np.zeros(len(obf_idx), dtype=int)

t0 = time.time()
for r, i in enumerate(obf_idx):
    a = shs[i]
    if not a:
        continue
    counts = defaultdict(int)
    for s in a:
        for j in inv.get(s, ()):
            counts[j] += 1
    n_cand[r] = len(counts)
    for j, c in counts.items():
        union = len(a) + len(shs[j]) - c
        jac = c / union if union else 0.0
        if jac > best_jac[r]:
            best_jac[r], best_mate[r] = jac, j
        con = c / len(a)
        if con > best_con[r]:
            best_con[r] = con
print(f'exact scan finished in {time.time() - t0:.1f}s')

# ---- positive control: the scanner must find a planted near-duplicate ----
probe_words = str(allsp.loc[int(ref_idx[0]), 'text']).split()
probe = ' '.join(probe_words[:max(int(len(probe_words) * 0.9), 6)])
pa, cnt = shingle_set(probe), defaultdict(int)
for s in pa:
    for j in inv.get(s, ()):
        cnt[j] += 1
ctrl = max((c / (len(pa) + len(shs[j]) - c) for j, c in cnt.items()), default=0.0)
print(f'positive control: planted 90% copy recovered at Jaccard {ctrl:.3f} '
      f'({"OK" if ctrl > 0.5 else "FAILED -- do not trust the zeros below"})')

res = {
    'method': 'exact Jaccard + containment over word 5-gram shingles via inverted index (no LSH threshold)',
    'k_shingle': K_SHINGLE,
    'n_obfuscated': int(len(obf_idx)),
    'n_reference_docs': int(len(ref_idx)),
    'n_obf_sharing_at_least_one_5gram': int((n_cand > 0).sum()),
    'max_jaccard': round(float(best_jac.max()), 6),
    'mean_jaccard': round(float(best_jac.mean()), 6),
    'p99_jaccard': round(float(np.quantile(best_jac, 0.99)), 6),
    'n_with_jaccard_ge_0.2': int((best_jac >= 0.2).sum()),
    'max_containment': round(float(best_con.max()), 6),
    'n_with_containment_ge_0.5': int((best_con >= 0.5).sum()),
    'positive_control_jaccard': round(float(ctrl), 4),
}
json.dump(res, open(OUT / 'pairing_check.json', 'w'), indent=2)
print('\n' + json.dumps(res, indent=2))

if res['max_jaccard'] < 0.05 and res['max_containment'] < 0.20 and ctrl > 0.5:
    print('\nCONCLUSION: the AI-obfuscated class is unpaired under an exact scan that would have '
          'found a partial match had one existed. No re-slicing of the released data can create '
          'pairs, so the within-item design below is the only way to answer the comment.')
else:
    print('\nCONCLUSION: some overlap exists -- inspect best_jac / best_con before writing the '
          'rebuttal, and consider recovering those pairs directly.')


## 4. Route A — recover native (source, paraphrase) pairs if the file provides them

In [ ]:
pairs_native = None

if not RUN_ROUTE_A:
    print('[Route A] skipped: RUN_ROUTE_A is False')
elif not TQ_PARAPHRASE_FILE:
    print('[Route A] skipped: TQ_PARAPHRASE_FILE is empty. Set it to the original '
          'TQ_SYNT_Identification_of_paraphrasing file to answer this comment at zero compute cost.')
else:
    p = Path(TQ_PARAPHRASE_FILE)
    if not p.exists():
        print('[Route A] file not found:', p)
    else:
        if p.suffix == '.json':
            raw = pd.DataFrame(json.load(open(p, encoding='utf-8')))
        elif p.suffix == '.jsonl':
            raw = pd.read_json(p, lines=True)
        elif p.suffix in ('.csv', '.tsv'):
            raw = pd.read_csv(p, sep='\t' if p.suffix == '.tsv' else ',')
        else:
            raw = pd.read_parquet(p)
        print('[Route A] columns available:', list(raw.columns))

        if SRC_FIELD not in raw.columns or TGT_FIELD not in raw.columns:
            print(f'[Route A] {SRC_FIELD!r} and/or {TGT_FIELD!r} not present. '
                  'Inspect the column list above and update SRC_FIELD / TGT_FIELD.')
        else:
            pn = raw[[SRC_FIELD, TGT_FIELD]].dropna().copy()
            pn.columns = ['source_text', 'paraphrase_text']
            pn['source_text']     = pn['source_text'].astype(str).str.strip()
            pn['paraphrase_text'] = pn['paraphrase_text'].astype(str).str.strip()
            pn = pn[(pn.source_text != '') & (pn.paraphrase_text != '')].reset_index(drop=True)

            # match back to the released obfuscated documents so the pairing is auditable
            def norm_key(t):
                return ''.join(ch for ch in str(t).lower() if ch.isalnum())

            released = set(allsp.loc[allsp.label == 2, 'text'].map(norm_key))
            pn['in_released_benchmark'] = pn['paraphrase_text'].map(lambda t: norm_key(t) in released)

            pn['src_words']   = pn['source_text'].str.split().str.len()
            pn['par_words']   = pn['paraphrase_text'].str.split().str.len()
            pn['delta_words'] = pn['par_words'] - pn['src_words']
            # how much was actually reworded, on the same scale as Part 0
            pn['self_jaccard'] = [jaccard(shingle_set(a), shingle_set(b))
                                  for a, b in zip(pn.source_text, pn.paraphrase_text)]

            pn.to_json(OUT / 'pairs_native.jsonl', orient='records', lines=True, force_ascii=False)
            pairs_native = pn

            print(f'\n[Route A] recovered {len(pn)} pairs; '
                  f'{int(pn.in_released_benchmark.sum())} match released obfuscated documents')
            print(pn[['src_words', 'par_words', 'delta_words', 'self_jaccard']].describe().to_string())

            from scipy.stats import wilcoxon as _wx
            try:
                _st, _p = _wx(pn.par_words, pn.src_words)
                _ptxt = f'Wilcoxon signed-rank p = {_p:.3g}'
            except ValueError:
                _ptxt = 'Wilcoxon not computable (all differences zero)'
            print(f'\nMedian length change under the corpus paraphraser: '
                  f'{pn.delta_words.median():+.1f} words ({_ptxt}).')
            print(f'Median source/paraphrase 5-gram Jaccard: {pn.self_jaccard.median():.3f} '
                  '-- near 0 means the paraphraser rewrites rather than edits.')
            print('This is the direct, within-item evidence for whether the paraphraser shortens '
                  'text, which is the mechanism behind the length confound in Major comment 1.')


## 5. Route B — sample the two arms and define the paraphrase prompt

In [ ]:
PARAPHRASE_PROMPT = (
    'Төмендегі мәтінді мағынасын толық сақтай отырып, басқаша сөзбен қайта жазыңыз. '
    'Сөйлемдердің құрылымы мен сөз таңдауын өзгертіңіз. '
    'Тек қайта жазылған мәтінді жазыңыз, ешқандай түсініктеме қоспаңыз.\n\n'
    'Мәтін:\n{text}\n\nҚайта жазылған мәтін:'
)

pool = {'test': test, 'dev': dev, 'train': train}[SAMPLE_FROM]

rows = []
for lbl, arm in [(0, 'human_to_obf'), (1, 'aigen_to_obf')]:
    sub = pool[pool.label == lbl]
    take = min(N_PER_ARM, len(sub))
    if take < N_PER_ARM:
        print(f'WARNING: only {take} documents of class {ID2LABEL[lbl]} in the {SAMPLE_FROM} '
              f'split; N_PER_ARM={N_PER_ARM} requested.')
    idx = rng.choice(sub.index.to_numpy(), size=take, replace=False)
    for i in idx:
        rows.append({'pair_id': f'{arm}__{pool.at[i, "doc_id"]}',
                     'arm': arm,
                     'doc_id': pool.at[i, 'doc_id'],
                     'orig_index': int(i),
                     'orig_label': lbl,
                     'orig_text': pool.at[i, 'text'],
                     'orig_words': int(pool.at[i, 'n_words'])})

arms = pd.DataFrame(rows)
assert arms.pair_id.is_unique, 'pair_id collision -- check doc_id construction'
arms.to_json(OUT / 'paired_inputs.jsonl', orient='records', lines=True, force_ascii=False)

print(arms.groupby('arm').size().to_string())
print('\nSource length (words) by arm:')
print(arms.groupby('arm')['orig_words'].describe()[['count', 'mean', '50%', 'max']].to_string())


## 6. Resumable generation

In [ ]:
# ---------------- resumable batched generation ----------------
GEN_PATH = OUT / 'paired_generations.jsonl'

def load_done(path):
    """Read completed generations, tolerating a partially written final line."""
    done, bad = {}, 0
    if Path(path).exists():
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    bad += 1
                    continue
                if r.get('pair_id') and str(r.get('paraphrase_text', '')).strip():
                    done[r['pair_id']] = r
    if bad:
        print(f'  ignored {bad} malformed line(s) in {Path(path).name}')
    return done

done = load_done(GEN_PATH)
todo = arms[~arms.pair_id.isin(done)].reset_index(drop=True)
print(f'{len(done)} already generated, {len(todo)} remaining')

if RUN_ROUTE_B and len(todo):
    if not PARAPHRASER_ID:
        raise ValueError(
            'PARAPHRASER_ID is empty. Set it to a Hugging Face instruct model you can actually '
            'access, and verify the repo id on huggingface.co first. Using a paraphraser from a '
            'different family than the corpus generator is a feature here, not a problem: it makes '
            'this an unseen-paraphraser test as well as a paired one.')

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    has_cuda = torch.cuda.is_available()
    print(f'torch {torch.__version__} | CUDA available: {has_cuda}'
          + (f' | {torch.cuda.get_device_name(0)}' if has_cuda else ''))
    if not has_cuda:
        print('WARNING: no GPU visible. Generation on CPU will be impractically slow -- '
              'switch the Colab runtime to a GPU before continuing.')

    quant = None
    if LOAD_IN_4BIT and has_cuda:
        try:
            from transformers import BitsAndBytesConfig
            import bitsandbytes  # noqa: F401
            quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                       bnb_4bit_compute_dtype=torch.float16,
                                       bnb_4bit_use_double_quant=True)
        except Exception as e:
            print(f'4-bit quantisation unavailable ({type(e).__name__}: {e}) -- loading in fp16')

    tok = AutoTokenizer.from_pretrained(PARAPHRASER_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'          # required for correct batched generation

    load_kwargs = {'device_map': 'auto'} if has_cuda else {}
    if quant is not None:
        load_kwargs['quantization_config'] = quant
    try:   # transformers >= 4.56 renamed torch_dtype -> dtype
        model = AutoModelForCausalLM.from_pretrained(
            PARAPHRASER_ID, dtype=torch.float16, **load_kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            PARAPHRASER_ID, torch_dtype=torch.float16, **load_kwargs)
    model.eval()
    model.generation_config.pad_token_id = tok.pad_token_id

    USED_CHAT_TEMPLATE = getattr(tok, 'chat_template', None) is not None

    def build_prompt(text):
        raw = PARAPHRASE_PROMPT.format(text=text)
        if USED_CHAT_TEMPLATE:
            try:
                return tok.apply_chat_template([{'role': 'user', 'content': raw}],
                                               tokenize=False, add_generation_prompt=True)
            except Exception:
                pass
        return raw

    todo = todo.copy()
    todo['prompt'] = [build_prompt(t) for t in todo['orig_text']]
    todo['src_tokens'] = [len(tok(t, add_special_tokens=False)['input_ids'])
                          for t in todo['orig_text']]
    todo['prompt_tokens'] = [len(tok(p, add_special_tokens=not USED_CHAT_TEMPLATE)['input_ids'])
                             for p in todo['prompt']]

    too_long = todo.prompt_tokens > MAX_SRC_TOKENS
    if too_long.any():
        print(f'excluding {int(too_long.sum())} source(s) longer than MAX_SRC_TOKENS='
              f'{MAX_SRC_TOKENS} rather than truncating them; logged to excluded_too_long.jsonl')
        todo[too_long].drop(columns=['prompt']).to_json(
            OUT / 'excluded_too_long.jsonl', orient='records', lines=True, force_ascii=False)
        todo = todo[~too_long].reset_index(drop=True)

    # longest sources first: a batch's budget is set by its longest member, and
    # sorting keeps batches homogeneous so short items do not wait on long ones
    todo = todo.sort_values('src_tokens', ascending=False).reset_index(drop=True)

    gen_meta = {'paraphraser_id': PARAPHRASER_ID, 'temperature': TEMPERATURE, 'top_p': TOP_P,
                'load_in_4bit': bool(quant is not None), 'seed': SEED,
                'chat_template': USED_CHAT_TEMPLATE,
                'new_token_multiplier': NEW_TOKEN_MULTIPLIER,
                'max_new_tokens_cap': MAX_NEW_TOKENS_CAP,
                'max_src_tokens': MAX_SRC_TOKENS}
    json.dump(gen_meta, open(OUT / 'generation_config.json', 'w'), indent=2, ensure_ascii=False)

    t0, n_cap = time.time(), 0
    with open(GEN_PATH, 'a', encoding='utf-8') as fout:
        for s in range(0, len(todo), GEN_BATCH):
            chunk = todo.iloc[s:s + GEN_BATCH]
            torch.manual_seed(SEED + s)          # reproducible, and stable across resumes

            budget = int(NEW_TOKEN_MULTIPLIER * int(chunk.src_tokens.max()) + NEW_TOKEN_SLACK)
            budget = max(64, min(budget, MAX_NEW_TOKENS_CAP))

            enc = tok(list(chunk['prompt']), return_tensors='pt', padding=True,
                      add_special_tokens=not USED_CHAT_TEMPLATE).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=budget, do_sample=True,
                                     temperature=TEMPERATURE, top_p=TOP_P,
                                     pad_token_id=tok.pad_token_id)
            new_ids = out[:, enc['input_ids'].shape[1]:]
            texts   = tok.batch_decode(new_ids, skip_special_tokens=True)
            # a sequence that never emitted EOS ran out of budget: its paraphrase is truncated
            stopped = (new_ids == tok.eos_token_id).any(dim=1).tolist()

            for (_, row), g, ok in zip(chunk.iterrows(), texts, stopped):
                rec = row.drop(labels=['prompt']).to_dict()
                rec['paraphrase_text'] = g.strip()
                rec['max_new_tokens']  = budget
                rec['hit_token_cap']   = (not bool(ok))
                rec['gen_timestamp']   = time.strftime('%Y-%m-%dT%H:%M:%S')
                n_cap += rec['hit_token_cap']
                fout.write(json.dumps(rec, ensure_ascii=False) + '\n')
            fout.flush()
            el = time.time() - t0
            print(f'  {min(s + GEN_BATCH, len(todo))}/{len(todo)}  '
                  f'budget={budget}  elapsed={el/60:.1f}min', flush=True)

    print(f'\n{n_cap} generation(s) hit the token cap and will be dropped during validation')
    del model
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

done = load_done(GEN_PATH)
print('total generated on file:', len(done))


## 7. Clean and validate the generations

In [ ]:
# ---------------- clean and validate the generations ----------------
gen = pd.DataFrame(list(done.values()))
if not len(gen):
    raise RuntimeError('No generations found. Run the generation cell, or check GEN_PATH.')

if 'hit_token_cap' not in gen.columns:      # files written by revision 1
    gen['hit_token_cap'] = False
    print('NOTE: this file predates the token-cap check; truncated paraphrases cannot be '
          'identified. Delete paired_generations.jsonl and regenerate for a clean Delta-words result.')

def clean_generation(t):
    t = str(t).strip()
    t = re.sub(r'^```[a-zA-Z]*\s*', '', t)
    t = re.sub(r'\s*```$', '', t)
    t = re.sub(r'^(Қайта жазылған мәтін|Қайта жазылған нұсқа|Мәтін|Жауап)\s*[:\-—]\s*', '', t)
    t = re.sub(r"^(Here is|Here's|Sure[,!])[^\n:]{0,60}:\s*", '', t, flags=re.I)
    t = t.strip().strip('"“”«»').strip()
    t = re.sub(r'[ \t]+\n', '\n', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

def cyrillic_ratio(t):
    """Share of Cyrillic among ALPHABETIC characters -- punctuation and digits do not dilute it."""
    letters = [c for c in str(t) if c.isalpha()]
    if not letters:
        return 0.0
    return sum('Ѐ' <= c <= 'ӿ' for c in letters) / len(letters)

gen['paraphrase_text'] = gen['paraphrase_text'].map(clean_generation)
gen['par_words']    = gen['paraphrase_text'].str.split().str.len().fillna(0).astype(int)
gen['delta_words']  = gen['par_words'] - gen['orig_words']
gen['cyr_ratio']    = gen['paraphrase_text'].map(cyrillic_ratio)
gen['self_jaccard'] = [jaccard(shingle_set(a), shingle_set(b))
                       for a, b in zip(gen.orig_text, gen.paraphrase_text)]

# ---- ordered rejection rules, with a printed accounting table ----
reason = pd.Series('kept', index=gen.index, dtype=object)
reason[gen.self_jaccard >= VERBATIM_JACCARD] = 'verbatim_copy'
reason[gen.par_words < MIN_PARAPHRASE_WORDS] = 'too_short'
reason[gen.hit_token_cap.astype(bool)]       = 'truncated_at_token_cap'
if DROP_LOW_CYRILLIC:
    reason[gen.cyr_ratio < MIN_CYRILLIC_RATIO] = 'script_drift'
gen['reject_reason'] = reason

print('validation outcome')
print(gen.groupby(['arm', 'reject_reason']).size().unstack(fill_value=0).to_string())

rejected = gen[gen.reject_reason != 'kept']
if len(rejected):
    rejected.to_json(OUT / 'paired_rejected.jsonl', orient='records', lines=True, force_ascii=False)
    # attrition is not random: long sources are the ones that hit the cap
    print(f'\nattrition check -- median source length, kept vs dropped:')
    print(f'  kept    {gen.loc[gen.reject_reason == "kept", "orig_words"].median():.0f} words '
          f'(n={int((gen.reject_reason == "kept").sum())})')
    print(f'  dropped {rejected.orig_words.median():.0f} words (n={len(rejected)})')
    print('  If dropped documents are systematically longer, say so in the paper: the paired '
          'sample is then biased towards shorter texts.')
if not DROP_LOW_CYRILLIC and (gen.cyr_ratio < MIN_CYRILLIC_RATIO).any():
    print(f'\nWARNING: {int((gen.cyr_ratio < MIN_CYRILLIC_RATIO).sum())} generations are '
          '<50% Cyrillic and were KEPT (DROP_LOW_CYRILLIC=False). A paraphraser that drifts out '
          'of Kazakh makes that arm uninterpretable.')

gen = gen[gen.reject_reason == 'kept'].drop(columns=['reject_reason']).reset_index(drop=True)
if not len(gen):
    raise RuntimeError('Every generation was rejected. Inspect paired_rejected.jsonl.')
gen.to_json(OUT / 'paired_clean.jsonl', orient='records', lines=True, force_ascii=False)

print(f'\n{len(gen)} usable pairs')
print(gen.groupby('arm')[['orig_words', 'par_words', 'delta_words', 'self_jaccard']]
      .median().round(3).to_string())
print('\nself_jaccard is the 5-gram overlap between source and paraphrase: near 0 means the '
      'paraphraser genuinely rewrote the text rather than editing a few words.')


## 8. Score each pair before and after

In [ ]:
import tensorflow as tf, keras
try:
    import keras_nlp
except ImportError:
    import keras_hub as keras_nlp
from scipy.special import softmax
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report

print(f'TF {tf.__version__} | Keras {keras.__version__}')
print('GPUs:', [d.name for d in tf.config.list_physical_devices('GPU')] or 'none')
try:
    if tf.config.list_logical_devices('TPU'):
        print('NOTE: TPU present. Pre-tokenising on CPU (as below) is what makes SentencePiece '
              'models usable here at all -- do not move the tokenizer back into tf.data.')
except Exception:
    pass

_REG = {'xlm_roberta': ('XLMRobertaClassifier', 'XLMRobertaPreprocessor'),
        'deberta_v3':  ('DebertaV3Classifier',  'DebertaV3Preprocessor'),
        'bert':        ('BertClassifier',       'BertPreprocessor'),
        'distil_bert': ('DistilBertClassifier', 'DistilBertPreprocessor')}
if CLF_FAMILY not in _REG:
    raise KeyError(f'CLF_FAMILY={CLF_FAMILY!r} unknown; choose one of {sorted(_REG)}')
_CLS, _PRE = _REG[CLF_FAMILY]

_PREPROC = None

def get_preprocessor(force_new=False):
    """One preprocessor, held at module level so its SentencePiece resource stays alive."""
    global _PREPROC
    if _PREPROC is None or force_new:
        _PREPROC = getattr(keras_nlp.models, _PRE).from_preset(CLF_PRESET, sequence_length=SEQ_LEN)
    return _PREPROC

def encode(texts, chunk=256, name=''):
    """Tokenise eagerly on CPU into NumPy arrays. SentencePiece never enters a tf.data graph."""
    texts = [str(t) for t in texts]
    parts = None
    for attempt in (0, 1):
        try:
            pre, parts = get_preprocessor(force_new=bool(attempt)), []
            with tf.device('/CPU:0'):
                for i in range(0, len(texts), chunk):
                    o = pre(tf.constant(texts[i:i + chunk], dtype=tf.string))
                    parts.append({k: np.asarray(v) for k, v in dict(o).items()})
            break
        except tf.errors.NotFoundError:
            if attempt:
                raise
            print('  SentencePiece resource was invalidated -- rebuilding the preprocessor, retrying')
    enc = {k: np.concatenate([p[k] for p in parts], axis=0) for k in parts[0]}
    print(f'  encoded {name or "texts"}: {len(texts)} docs -> '
          f'{tuple(next(iter(enc.values())).shape)}')
    return enc

def build_clf():
    """Classifier WITHOUT a preprocessor: it consumes token ids directly."""
    try:
        return getattr(keras_nlp.models, _CLS).from_preset(
            CLF_PRESET, num_classes=3, preprocessor=None)
    except Exception as e:
        raise RuntimeError(
            f'Could not load preset {CLF_PRESET!r} ({type(e).__name__}: {e}). '
            'Presets download from Kaggle -- check network access and, if prompted, '
            'KAGGLE_USERNAME / KAGGLE_KEY.') from e

def make_ds(X, y=None, batch=32, shuffle_seed=None):
    n = len(next(iter(X.values())))
    ds = tf.data.Dataset.from_tensor_slices((X, y) if y is not None else X)
    if shuffle_seed is not None:
        ds = ds.shuffle(n, seed=shuffle_seed, reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def slug(s):
    return s.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')

y_train = train['label'].to_numpy('int32')
y_dev   = dev['label'].to_numpy('int32')
cw = compute_class_weight('balanced', classes=np.array(LABELS), y=y_train)
class_weight = {int(k): float(v) for k, v in zip(LABELS, cw)}
print('class weights:', {ID2LABEL[k]: round(v, 3) for k, v in class_weight.items()})

# ---- encode EVERYTHING up front, before any model exists ----
gen = pd.read_json(OUT / 'paired_clean.jsonl', lines=True)
print(f'\nencoding {len(gen)} pairs plus the held-out test split')
ENC = {'orig': encode(gen['orig_text'], name='pair sources'),
       'para': encode(gen['paraphrase_text'], name='paraphrases'),
       'test': encode(test['text'], name='test split')}
X_train = X_dev = None

def get_model(seed):
    """Load seed weights if present; otherwise train once and SAVE them."""
    global X_train, X_dev
    wp = Path(WEIGHTS_DIR) / f'{slug(CLF_MODEL)}__seed{seed}.weights.h5'
    clf = build_clf()
    clf.compile(optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE,
                                                 weight_decay=WEIGHT_DECAY),
                loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
    if wp.exists():
        clf.load_weights(str(wp))
        print(f'  loaded weights {wp.name}')
        return clf
    if not TRAIN_IF_MISSING:
        raise FileNotFoundError(
            f'{wp} not found and TRAIN_IF_MISSING is False. Run notebook 09 first.')
    print(f'  no saved weights for seed {seed} -- training now (run notebook 09 to avoid this)')
    if X_train is None:
        X_train, X_dev = encode(train['text'], name='train'), encode(dev['text'], name='dev')
    keras.utils.set_random_seed(seed)
    clf.fit(make_ds(X_train, y_train, batch=BATCH_SIZE, shuffle_seed=seed),
            validation_data=make_ds(X_dev, y_dev, batch=BATCH_SIZE),
            epochs=EPOCHS, class_weight=class_weight,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                                     restore_best_weights=True)],
            verbose=1)
    wp.parent.mkdir(parents=True, exist_ok=True)
    clf.save_weights(str(wp))
    print(f'  saved {wp.name} -- the next run will reuse it')
    return clf

# ---- one pass over the seeds; every seed scores every set before release ----
probs = {k: None for k in ENC}
t0 = time.time()
for sd in CLF_SEEDS:
    print(f'[seed {sd}]')
    clf = get_model(sd)
    for k, xx in ENC.items():
        p = softmax(np.asarray(clf.predict(make_ds(xx, batch=PREDICT_BATCH), verbose=0)), axis=1)
        probs[k] = p if probs[k] is None else probs[k] + p
    del clf
    gc.collect()
    keras.backend.clear_session()     # safe: no SentencePiece lives in any graph
probs = {k: v / len(CLF_SEEDS) for k, v in probs.items()}
print(f'ensemble of {len(CLF_SEEDS)} seeds scored in {(time.time() - t0)/60:.1f} min')

# ---- sanity anchor: the ensemble must reproduce the manuscript's headline number ----
y_test = test['label'].to_numpy()
p_test = probs['test'].argmax(1)
macro_f1 = f1_score(y_test, p_test, average='macro')
print(f'\nheld-out test macro-F1 (ensemble) = {macro_f1:.4f} | accuracy = '
      f'{accuracy_score(y_test, p_test):.4f}')
print(classification_report(y_test, p_test, target_names=[ID2LABEL[i] for i in LABELS], digits=3))
if macro_f1 < 0.70:
    print('WARNING: this is far below the manuscript figure. The weights, the preset or the label '
          'mapping is wrong -- do not interpret the paired tables below until this is resolved.')

# ---- attach predictions to the pairs ----
for side, key in [('before', 'orig'), ('after', 'para')]:
    gen[f'pred_{side}']      = probs[key].argmax(1)
    gen[f'conf_{side}']      = probs[key].max(1)
    gen[f'p_machine_{side}'] = probs[key][:, 1] + probs[key][:, 2]   # P(AI-Gen) + P(AI-Obf)
np.save(OUT / 'probs_orig.npy', probs['orig'])
np.save(OUT / 'probs_para.npy', probs['para'])
gen.to_json(OUT / 'paired_scored.jsonl', orient='records', lines=True, force_ascii=False)
print('scored', len(gen), 'pairs ->', (OUT / "paired_scored.jsonl").name)


## 9. Paired analyses (McNemar, transition matrices)

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar as sm_mcnemar

def binary(pred):
    """Operational mapping from Section III-B: {AI-Generated, AI-Obfuscated} -> machine."""
    return (np.asarray(pred) >= 1).astype(int)

def boot_ci(x, n=BOOTSTRAP_N, alpha=0.05, seed=0):
    """Percentile bootstrap CI for a mean (x may be a 0/1 indicator)."""
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (float('nan'), float('nan'))
    r = np.random.default_rng(seed)
    means = x[r.integers(0, len(x), size=(n, len(x)))].mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

rows = []
for arm, g in gen.groupby('arm'):
    orig_lbl = int(g['orig_label'].iloc[0])
    b_before, b_after = binary(g['pred_before']), binary(g['pred_after'])

    to_human   = (b_before == 1) & (b_after == 0)   # machine -> human  (evasion)
    to_machine = (b_before == 0) & (b_after == 1)   # human   -> machine

    lo_b, hi_b = boot_ci(b_before, seed=1)
    lo_a, hi_a = boot_ci(b_after,  seed=2)

    row = {
        'arm': arm, 'n': len(g), 'orig_class': ID2LABEL[orig_lbl],
        'pct_machine_before': round(100 * b_before.mean(), 2),
        'ci_machine_before': f'[{100*lo_b:.1f}, {100*hi_b:.1f}]',
        'pct_machine_after': round(100 * b_after.mean(), 2),
        'ci_machine_after': f'[{100*lo_a:.1f}, {100*hi_a:.1f}]',
        'pct_flip_to_human': round(100 * to_human.mean(), 2),
        'pct_flip_to_machine': round(100 * to_machine.mean(), 2),
        'pct_3way_before_correct': round(100 * float((g.pred_before == orig_lbl).mean()), 2),
        'pct_3way_after_obfuscated': round(100 * float((g.pred_after == 2).mean()), 2),
        'pct_3way_after_human': round(100 * float((g.pred_after == 0).mean()), 2),
        'mean_p_machine_before': round(float(g.p_machine_before.mean()), 4),
        'mean_p_machine_after': round(float(g.p_machine_after.mean()), 4),
        'median_delta_words': float(g.delta_words.median()),
        'median_self_jaccard': round(float(g.self_jaccard.median()), 4),
    }
    row['interpretation'] = ('false-positive rate -> detection rate' if orig_lbl == 0
                             else 'detection rate; 100 - after = evasion rate')

    # McNemar on the CHANGE in the binary decision (the paired quantity)
    n01, n10 = int(to_human.sum()), int(to_machine.sum())
    row['mcnemar_machine_to_human'] = n01
    row['mcnemar_human_to_machine'] = n10
    if n01 + n10 > 0:
        tbl = [[int(((b_before == 1) & (b_after == 1)).sum()), n01],
               [n10, int(((b_before == 0) & (b_after == 0)).sum())]]
        r = sm_mcnemar(tbl, exact=(n01 + n10 < 25), correction=True)
        row['mcnemar_p'] = float(f'{r.pvalue:.3g}')
    else:
        row['mcnemar_p'] = float('nan')

    # Wilcoxon on the paired shift in P(machine) -- uses magnitude, not just sign
    try:
        _, pw = wilcoxon(g.p_machine_after, g.p_machine_before)
        row['wilcoxon_p_machine_p'] = float(f'{pw:.3g}')
    except ValueError:
        row['wilcoxon_p_machine_p'] = float('nan')
    # Wilcoxon on the paired change in length
    try:
        _, pl = wilcoxon(g.par_words, g.orig_words)
        row['wilcoxon_delta_words_p'] = float(f'{pl:.3g}')
    except ValueError:
        row['wilcoxon_delta_words_p'] = float('nan')

    rows.append(row)

paired_tbl = pd.DataFrame(rows)
paired_tbl.to_csv(OUT / 'table_paired_paraphrase.csv', index=False)

main_cols = ['arm', 'n', 'orig_class', 'pct_machine_before', 'ci_machine_before',
             'pct_machine_after', 'ci_machine_after', 'pct_flip_to_human',
             'pct_flip_to_machine', 'mcnemar_p', 'wilcoxon_p_machine_p', 'median_delta_words']
print('=== paired results (binary machine/human decision) ===')
print(paired_tbl[main_cols].to_string(index=False))
print('\n=== three-way detail ===')
print(paired_tbl[['arm', 'pct_3way_before_correct', 'pct_3way_after_obfuscated',
                  'pct_3way_after_human', 'median_self_jaccard',
                  'wilcoxon_delta_words_p']].to_string(index=False))

# ---- three-way transition matrices ----
names = [ID2LABEL[i] for i in LABELS]
for arm, g in gen.groupby('arm'):
    cm = (pd.crosstab(g['pred_before'].map(ID2LABEL), g['pred_after'].map(ID2LABEL))
            .reindex(index=names, columns=names, fill_value=0))
    cm.index.name, cm.columns.name = 'before', 'after'
    print(f'\n=== {arm}: predicted class before -> after paraphrasing ===')
    print(cm.to_string())
    cm.to_csv(OUT / f'transition_{arm}.csv')

# ---- do the generated paraphrases look like the corpus ones? ----
corpus_obf = test[test.label == 2]
print(f'\nCorpus AI-obfuscated median length: {corpus_obf.n_words.median():.0f} words')
print(gen.groupby('arm')['par_words'].median().to_string())
print('\nIf your generated paraphrases differ sharply in length from the corpus ones, or are '
      'classified as AI-obfuscated far less often, then the reported F1 reflects the CORPUS '
      'PIPELINE rather than paraphrasing in general. That is the single most important thing this '
      'notebook can tell you, and it speaks directly to what the AI-obfuscated class represents.')


## 10. Figure — decision shift and length change

In [ ]:
# ---------------- figure: decision shift + length change ----------------
# Reuses binary() and boot_ci() from the cell above.
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 9, 'figure.dpi': 120, 'savefig.bbox': 'tight'})

arms = [a for a in ['human_to_obf', 'aigen_to_obf'] if a in set(gen.arm)]
labels = {'human_to_obf': 'Human\n-> paraphrased',
          'aigen_to_obf': 'AI-generated\n-> paraphrased'}
ticks = [labels[a] for a in arms]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.4))

# (a) share classified as "machine", before vs after
x = np.arange(len(arms))
for i, side in enumerate(['before', 'after']):
    pct, err = [], []
    for j, a in enumerate(arms):
        b = binary(gen.loc[gen.arm == a, f'pred_{side}'])
        lo, hi = boot_ci(b, seed=10 * i + j)
        pct.append(100 * b.mean())
        err.append([100 * b.mean() - 100 * lo, 100 * hi - 100 * b.mean()])
    err = np.array(err).T
    ax1.bar(x + (i - 0.5) * 0.36, pct, 0.34, yerr=err, capsize=3, label=side.capitalize())
    for xi, v, e in zip(x + (i - 0.5) * 0.36, pct, err[1]):
        ax1.text(xi, v + e + 2, f'{v:.0f}', ha='center', fontsize=8)

ax1.set_xticks(x)
ax1.set_xticklabels(ticks, fontsize=8)
ax1.set_ylabel('classified as machine (%)')
ax1.set_ylim(0, 115)
ax1.set_title('(a) Decision shift under paraphrasing', loc='left')
ax1.legend(frameon=False, fontsize=8)

# (b) paired change in length
data = [gen.loc[gen.arm == a, 'delta_words'].to_numpy(float) for a in arms]
ax2.boxplot(data, widths=0.45, showfliers=False)
ax2.axhline(0, color='grey', ls='--', lw=1)
for i, dd in enumerate(data, start=1):
    ax2.text(i, np.percentile(dd, 75) + 2, f'median {np.median(dd):+.0f}',
             ha='center', fontsize=8)
ax2.set_xticks(range(1, len(arms) + 1))
ax2.set_xticklabels(ticks, fontsize=8)
ax2.set_ylabel('delta words (paraphrase - source)')
ax2.set_title('(b) Within-pair length change', loc='left')

fig.tight_layout()
# PNG only: the PDF backend needs a matplotlib font module that a partially
# upgraded Colab environment does not always provide.
fig.savefig(OUT / 'fig_paired_paraphrase.png', dpi=300)
plt.show()
print('saved fig_paired_paraphrase.png to', OUT)

# same numbers as a table, for the caption and for accessibility
print()
print(paired_tbl[['arm', 'pct_machine_before', 'ci_machine_before',
                  'pct_machine_after', 'ci_machine_after',
                  'median_delta_words']].to_string(index=False))


## 11. Run manifest

In [ ]:
# ---------------- run manifest: what produced these numbers ----------------
import importlib

def _v(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return 'not installed'

manifest = {
    'notebook': '07_paired_paraphrase_design.ipynb (revision 2)',
    'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'wall_clock_minutes': round((time.time() - T_START) / 60, 2),
    'python': platform.python_version(),
    'versions': {m: _v(m) for m in ['numpy', 'pandas', 'scipy', 'sklearn', 'tensorflow',
                                    'keras', 'keras_hub', 'transformers', 'torch',
                                    'statsmodels', 'matplotlib']},
    'config': {
        'REPO_DIR': str(REPO), 'RESULTS_DIR': str(OUT), 'WEIGHTS_DIR': str(WEIGHTS_DIR),
        'SEED': SEED, 'SAMPLE_FROM': SAMPLE_FROM, 'N_PER_ARM': N_PER_ARM,
        'PARAPHRASER_ID': PARAPHRASER_ID, 'LOAD_IN_4BIT': LOAD_IN_4BIT,
        'TEMPERATURE': TEMPERATURE, 'TOP_P': TOP_P,
        'NEW_TOKEN_MULTIPLIER': NEW_TOKEN_MULTIPLIER, 'MAX_NEW_TOKENS_CAP': MAX_NEW_TOKENS_CAP,
        'MAX_SRC_TOKENS': MAX_SRC_TOKENS,
        'CLF_MODEL': CLF_MODEL, 'CLF_PRESET': CLF_PRESET, 'CLF_SEEDS': CLF_SEEDS,
        'SEQ_LEN': SEQ_LEN, 'EPOCHS': EPOCHS, 'BATCH_SIZE': BATCH_SIZE,
        'LEARNING_RATE': LEARNING_RATE, 'WEIGHT_DECAY': WEIGHT_DECAY,
        'MIN_CYRILLIC_RATIO': MIN_CYRILLIC_RATIO, 'DROP_LOW_CYRILLIC': DROP_LOW_CYRILLIC,
        'VERBATIM_JACCARD': VERBATIM_JACCARD, 'BOOTSTRAP_N': BOOTSTRAP_N,
    },
    'results': {
        'n_pairs_scored': int(len(gen)),
        'test_macro_f1_ensemble': round(float(macro_f1), 4),
        'pairing_check': res,
    },
    'outputs': sorted(p.name for p in OUT.iterdir() if p.is_file()),
}
json.dump(manifest, open(OUT / 'run_manifest.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps({k: manifest[k] for k in ['finished_at', 'wall_clock_minutes', 'results']},
                 indent=2, ensure_ascii=False))
print('\nfiles in', OUT)
for f in manifest['outputs']:
    print('  ', f)
